# [1교시]

### LangChain을 활용한 ReAct 에이전트

### 1. LangChain Agent 프레임워크 소개
LangChain은 에이전트를 쉽게 구축할 수 있도록 추상화된 클래스들을 제공합니다. 최신 버전에서는 내부적으로 LangGraph 기반의 강력한 상태 관리 엔진을 사용합니다.
- **LLM/ChatModel**: 추론을 담당하는 엔진 (예: `ChatOpenAI`)
- **Tools**: 에이전트가 외부 세계와 상호작용할 수 있는 기능의 집합
- **create_agent**: LLM과 도구를 결합하여 알아서 ReAct 사이클(Thought -> Action -> Observation)을 수행하는 지능형 에이전트 객체를 반환합니다.

### 2. Custom Tool 정의
@tool 데코레이터를 이용해 아주 쉽게 파이썬 함수를 에이전트가 사용할 수 있는 도구로 만들 수 있습니다.

In [1]:
!pip install langchain-openai langchain-core python-dotenv

In [2]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from dotenv import load_dotenv
load_dotenv(override=True)

# Tool 정의(@tool 데코레이션 사용)
@tool
def search_knowledge(query:str) -> str:
    '''지식베이스를 검색합니다. 모르는 정보가 있을 때 사용하세요'''
    kb = {
        'LangChain' : ''
    }
    
    for key, val in kb.items():
        if key.lower() in query.lower():
            return val
    return '관련 정보를 찾을 수 없습니다.'
@tool
def string_length(text:str)->int:
    '''문자열 길이를 반환합니다. 텍스트의 글자 수를 알아야 할 때 사용하세요'''
    return len(text)

@tool
def calculate_discount(price:float, discount_rate:float)->float:
    '''원래 가격(price)과 할인율(discount_rate, 예 20%면 0.2)을 입력받아서 할인가격을 계산합니다.'''
    return price * (1-discount_rate)
tools = [search_knowledge, string_length, calculate_discount]

### 3. ReAct 에이전트 생성 및 실행
최신 LangChain에서는 `create_agent` 함수 하나로 도구와 모델을 바인딩한 완성형 에이전트를 만들 수 있습니다.

In [3]:
from langchain.agents import create_agent
# chat모델 초기화
llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0)
# 에이전트 생성
agent = create_agent(llm, tools)

### 4. 에이전트 실행 및 추적
invoke 메서드를 호출하면 에이전트는 내부적으로 Thought -> Action -> Observation 과정을 거쳐 정답을 도출합니다. 결과는 메시지 리스트 형태로 반환됩니다.

In [4]:
# 단일도구사용
response1 = agent.invoke({'messages':[('user', '15000원짜리 물건을 15% 할인받으면 얼마인가요')]})
# 다중도구사용
response2 = agent.invoke({'messages':[('user', 
    'LangChain이라는 단어의 글자 수는 몇개인가요 그리고 지식베이스에서 LangChain이 무엇인지 검색해주세요')]})

print(f"단일도구사용\n{response1['messages'][-1].content}")
print(f"다중도구사용\n{response2['messages'][-1].content}")

단일도구사용
15,000원에서 15% 할인하면 **12,750원**입니다.
다중도구사용
- **“LangChain” 글자 수:** **9개**
- **지식베이스 검색 결과:** 현재 지식베이스에서 **LangChain의 정의/설명 내용을 찾지 못했습니다**(검색 결과가 비어 있습니다).


In [5]:
response3 = agent.invoke({'messages':[('system','너는 부동산 최신 뉴스 전문 수집 및 분석가야' ),
                                      ('user','금천구 가산동에서 매매된 가장 최신의 부동산 실거래 가격을 알려주세요')  ]})
print(response3['messages'][-1].content)

죄송하지만, 현재 제 환경에서는 **“금천구 가산동”의 “가장 최신” 매매 실거래가(개별 거래 단위 가격)**를 바로 조회해 확정적으로 제공할 수 있는 실거래 데이터 소스에 접근이 되지 않아, 정확한 최신 금액을 특정해 드릴 수 없습니다.

대신, 아래 정보를 주시면 **원하시는 형태(최신 순/평형/면적/단지별 등)**로 정리하는 방식으로 도와드릴게요.

1) 원하시는 범위: **아파트 / 오피스텔 / 빌라(다세대) / 상가** 중 어떤 유형인가요?  
2) “가장 최신” 기준: **최근 1개월 / 최근 3개월 / 올해 / 특정 날짜(예: 2026-05-중)** 중 무엇인가요?  
3) 면적 기준: **전용면적(예: 59㎡, 84㎡)** 또는 상관없음?

또는, 사용 중인 사이트/데이터(예: **국토교통부 실거래가 공개시스템** 링크나 검색 결과 화면 캡처/텍스트)를 주시면 그 안에서 **가장 최신 거래 1건(또는 TOP N)**을 골라 **가격 요약**까지 바로 정리해 드리겠습니다.


### 실시간 주식 매수,매도 확인 tool 제작
- 혼자 만들어보기

In [6]:
!pip install yfinance

In [7]:
import os
import yfinance as yf
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from dotenv import load_dotenv

load_dotenv(override=True)

# 코스피 TOP 10 종목 정보 매핑 사전
KOSPI_TOP10 = {
    '삼성전자': '005930.KS',
    'sk하이닉스': '000660.KS', '하이닉스': '000660.KS',
    'lg에너지솔루션': '373220.KS', '엘지에너지솔루션': '373220.KS', '엔솔': '373220.KS',
    '삼성바이오로직스': '207940.KS', '삼바': '207940.KS',
    '현대차': '005380.KS',
    '기아': '000270.KS',
    '셀트리온': '068270.KS',
    'kb금융': '105560.KS', '케イビー금융': '105560.KS',
    'posco홀딩스': '005490.KS', '포스코홀딩스': '005490.KS', '포스코': '005490.KS',
    'naver': '035420.KS', '네이버': '035420.KS'
}

@tool
def get_kospi_live_price(stock_name_or_ticker: str) -> str:
    """코스피 주식의 현재 실시간 가격을 조회합니다.
    입력값(stock_name_or_ticker)은 '삼성전자', '현대차' 같은 종목명이나 '005930.KS' 같은 티커 형태 모두 가능합니다.
    """
    # 1. 입력값이 종목명인 경우 사전에서 티커 검색
    clean_input = stock_name_or_ticker.strip().lower()
    ticker = KOSPI_TOP10.get(clean_input, stock_name_or_ticker)
    
    # 2. 만약 숫자로만 된 6자리 종목번호가 들어왔을 경우를 대비해 .KS 보정
    if clean_input.isdigit() and len(clean_input) == 6:
        ticker = f"{clean_input}.KS"

    try:
        stock = yf.Ticker(ticker)
        todays_data = stock.history(period='1d', interval='1m')
        
        if not todays_data.empty:
            current_price = float(todays_data['Close'].iloc[-1])
        else:
            current_price = float(stock.info.get('regularMarketPrice', 0))
            
        if current_price == 0:
            return f"'{stock_name_or_ticker}'의 주가 데이터를 가져올 수 없습니다. 올바른 종목명이나 티커인지 확인해주세요."
            
        return str(current_price)
        
    except Exception as e:
        return f"가격을 가져오는 중 오류가 발생했습니다: {e}"


@tool
def calculate_buy_profit(buy_price: float, current_price: float, quantity: int) -> str:
    """[매수 기준] 주식을 보유 중인 상태에서 실시간 수익을 계산합니다.
    - buy_price: 주식을 산 평단가
    - current_price: 현재 주가
    - quantity: 보유 주식 수
    """
    total_buy_amount = buy_price * quantity       # 총 매수 금액
    total_current_amount = current_price * quantity # 현재 가치
    
    profit_loss = total_current_amount - total_buy_amount # 현재 수익금
    profit_rate = (profit_loss / total_buy_amount) * 100 if total_buy_amount != 0 else 0
    
    return (
        f"[매수 보유 중인 주식 평가]\n"
        f"- 총 매수 금액: {total_buy_amount:,.0f}원 ({quantity}주)\n"
        f"- 현재 실시간 가치: {total_current_amount:,.0f}원\n"
        f"- 실시간 평가손익: {profit_loss:,.0f}원\n"
        f"- 실시간 수익률: {profit_rate:.2f}%"
    )


@tool
def calculate_sell_profit(buy_price: float, sell_price: float, quantity: int) -> str:
    """[매도 기준] 주식을 이미 팔아서 확정된 실시간 수익을 계산합니다.
    - buy_price: 주식을 샀던 평단가
    - sell_price: 주식을 판 가격
    - quantity: 매도한 주식 수
    """
    total_buy_amount = buy_price * quantity   # 투자했던 금액
    total_sell_amount = sell_price * quantity # 매도해서 확보한 금액
    
    realized_profit = total_sell_amount - total_buy_amount # 확정 수익금
    profit_rate = (realized_profit / total_buy_amount) * 100 if total_buy_amount != 0 else 0
    
    return (
        f"[매도 완료 - 수익 확정 브리핑]\n"
        f"- 총 매수 금액: {total_buy_amount:,.0f}원\n"
        f"- 총 매도 금액: {total_sell_amount:,.0f}원 ({quantity}주)\n"
        f"- 실시간 실현손익: {realized_profit:,.0f}원\n"
        f"- 실시간 수익률: {profit_rate:.2f}%"
    )

# 에이전트 툴 리스트
tools = [get_kospi_live_price, calculate_buy_profit, calculate_sell_profit]

In [8]:
from langgraph.prebuilt import create_react_agent

# 에이전트가 참고할 가이드라인 정의
system_prompt = (
    "당신은 국내 주식(코스피 TOP 10) 투자 수익 계산 전문가입니다. "
    "사용자가 종목 이름만 말해도 'get_kospi_live_price' 툴을 이용해 실시간 가격을 조회할 수 있습니다. "
    "사용자가 주식을 산 상태라면 'calculate_buy_profit'을, 이미 판 상태라면 'calculate_sell_profit'을 호출하세요. "
    "최종 답변을 줄 때는 수익률(%)과 종목 수량에 비례한 이득/손해 금액을 가독성 좋게 출력하세요."
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 에이전트에 시스템 메시지와 툴 등록
agent = create_react_agent(llm, tools, prompt=system_prompt)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_31784\1047817920.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=system_prompt)


In [9]:
query1 = "나 네이버 180,000원에 50주 매수해서 가지고 있는데 지금 실시간으로 이득이야 손해야? 얼마인지 퍼센트랑 같이 알려줘"
response1 = agent.invoke({'messages': [('user', query1)]})
print(response1['messages'][-1].content)

현재 네이버 주식은 234,000원이며, 50주를 180,000원에 매수하셨습니다. 

- **총 매수 금액**: 9,000,000원
- **현재 실시간 가치**: 11,700,000원
- **실시간 평가손익**: 2,700,000원
- **실시간 수익률**: 30.00%

따라서 현재 이득입니다!


In [10]:
query2 = "내가 예전에 SK하이닉스를 150,000원에 20주 샀었거든? 그걸 현재 가격에 전부 매도했어. 이득 본 결과 정산 좀 해줘"
response2 = agent.invoke({'messages': [('user', query2)]})
print(response2['messages'][-1].content)

SK하이닉스를 매도한 결과는 다음과 같습니다:

- **총 매수 금액**: 3,000,000원
- **총 매도 금액**: 46,390,000원 (20주)
- **실현 손익**: 43,390,000원
- **실현 수익률**: 1446.33%

매도하신 것에 대해 축하드립니다! 🎉


# [2교시]

## 1. 커스텀 Tool 설계 원칙

ReAct 에이전트에서 Tool은 에이전트가 외부 세계와 상호작용하는 유일한 통로이다. 따라서 Tool의 설계 품질이 에이전트의 전체 성능을 좌우한다. 다음은 커스텀 Tool을 설계할 때 반드시 지켜야 할 핵심 원칙들이다.

### 1.1 단일 책임 원칙 (Single Responsibility)

하나의 Tool은 **하나의 명확한 기능**만 수행해야 한다. 뉴스 검색과 요약을 하나의 Tool에 넣는 것이 아니라, 검색 Tool과 요약 Tool을 분리하여 구현한다. 이렇게 하면 LLM이 각 도구의 역할을 명확히 이해할 수 있고, 도구 조합의 유연성이 높아진다.

### 1.2 명확한 Tool 설명문 작성

Tool의 설명문(description)은 LLM이 도구를 올바르게 선택하는 데 결정적인 역할을 한다. 설명문에는 다음 요소가 포함되어야 한다:

- **기능 설명**: 이 도구가 무엇을 하는지 한 문장으로 명시
- **입력 형식**: 어떤 형식의 입력을 기대하는지 구체적으로 명시
- **사용 예시**: 실제 호출 예시를 포함하여 LLM이 패턴을 학습하도록 유도

### 1.3 입력/출력 형식 표준화

모든 Tool은 **문자열 입력 -> 문자열 출력** 형태를 유지하는 것이 좋다. ReAct 프레임워크에서 Action과 Observation은 텍스트 기반이므로, 일관된 문자열 인터페이스를 유지하면 에이전트와의 통합이 용이하다.

### 1.4 에러 처리와 Fallback 전략

외부 API를 호출하는 Tool은 반드시 예외 처리를 포함해야 한다. 네트워크 오류, 타임아웃, API 제한 등 다양한 실패 상황에 대비하여:

- try/except로 예외를 포착하고 의미 있는 오류 메시지를 반환
- 타임아웃 설정으로 무한 대기 방지
- 재시도 로직으로 일시적 오류에 대응
- 대체 데이터 소스나 기본값을 준비

## 2. 뉴스 검색 커스텀 Tool 개발

첫 번째 커스텀 Tool로 Google News RSS 피드를 활용한 **뉴스 검색 Tool**을 구현한다. 이 Tool은 실시간 뉴스 데이터를 수집하여 에이전트에게 최신 정보를 제공하는 역할을 한다.

### RSS 기반 뉴스 검색의 장점

- 별도의 API 키 없이 무료로 사용 가능
- XML 형식으로 구조화된 데이터 제공
- 실시간 최신 뉴스 접근 가능
- 한국어 뉴스 지원


- https://news.google.com/rss/search?q={query}&hl=ko&gl=KR&ceid=KR:ko

In [11]:
import os
import json
import requests
import xml.etree.ElementTree as ET
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def fetch_news(query, max_results=3):
    '''Goole News RSS에서 뉴스를 검색합니다.'''
    try:
        url = f'https://news.google.com/rss/search?q={query}&hl=ko&gl=KR&ceid=KR:ko'
        response = requests.get(url,timeout=10)        

        root = ET.fromstring(response.content)
        items = root.findall('.//item')[:max_results]  # .현재노드  // 모든 하위 태그탐색  item  <item>태그
        results = []
        for item in items:
            title = item.find('title').text if item.find('title') is not None else 'N/A'
            pub_date = item.find('pubDate').text if item.find('title') is not None else 'N/A'
            source = item.find('source').text if item.find('title') is not None else 'N/A'
            results.append({'title':title, 'date':pub_date, 'source':source})
        if results:
            output = f'{query}관련 최신 뉴스 {len(results)}건\n'
            for i, r in enumerate(results,1):
                output += f"  {i}. [{r['source']} {r['title']}  {r['date']}]"
            return output
        return f'{query}에 대한 뉴스를 찾을 수 없습니다'
    except Exception as e:
        return f'뉴스검색오류 : {e}'

print(fetch_news("지하철"))

지하철관련 최신 뉴스 3건
  1. [경향신문 [속보]지하철 홍대입구역~을지로입구역 운행 재개 - 경향신문  Thu, 28 May 2026 15:00:00 GMT]  2. [전자신문 “잠든 여성들만 노렸다”…런던 심야 지하철 연쇄 성범죄자 결국 실형 - 전자신문  Fri, 29 May 2026 04:00:00 GMT]  3. [연합뉴스 광주 지하철 공사장서 가스 누출 추정…작업자들 어지럼증 호소 - 연합뉴스  Fri, 29 May 2026 02:16:26 GMT]


# 요약이나 분석은 별도의 tool에 위임

In [12]:
# 텍스트 요약
def summarize_text(text: str, max_length=100):
    try:
        response = client.chat.completions.create(
            model='gpt-5.4-nano',
            # 1. messages 리스트는 여기서 깔끔하게 끝납니다.
            messages=[
                {'role': 'system', 'content': f'텍스트를 {max_length}자 이내로 간결하게 요약하는 시스템 입니다'},
                {'role': 'user', 'content': text}
            ],
            # 2. 파라미터들은 리스트 바깥, create() 함수 내부에 위치해야 합니다.
            temperature=0,
            max_completion_tokens=max_length + 100
        )
        return f'요약 : {response.choices[0].message.content}'
    except Exception as e:
        return f'요약 오류 : {e}'

In [13]:
# tool chain 테스트
summarize_text(fetch_news("인공지능"))

'요약 : AI 보안 강화: 정부 2027 독자 체계 구축, 민간 사이버위협 대응 총력, 스노우플레이크 나토마 인수로 AI 에이전트 보안·통제 강화.'

## 3. 에러 핸들링이 포함된 견고한 ReAct 에이전트

이전 에서 구현한 기본 ReAct 에이전트는 에러 발생 시 바로 중단되는 문제가 있었다. 실무 환경에서는 네트워크 장애, API 제한, 잘못된 입력 등 다양한 예외 상황이 발생할 수 있으므로, 이를 우아하게 처리하는 **견고한(Robust) 에이전트**가 필요하다.

### RobustReActAgent의 핵심 기능

- **자동 재시도(Retry)**: Tool 실행 실패 시 설정된 횟수만큼 재시도
- **실행 로그(Execution Log)**: 모든 단계를 기록하여 디버깅과 분석 지원
- **형식 오류 복구**: LLM이 잘못된 형식으로 응답하면 재시도 요청
- **최대 단계 제한**: 무한 루프를 방지하는 안전장치

# [3교시]

In [14]:
import re
import time
from typing import Callable, Dict, Optional
class RobustReActAgent:
    def __init__(self,model = 'gpt-5.4-nano', max_steps=6, retry_count=2):
        self.model=model
        self.max_steps = max_steps
        self.retry_count = retry_count
        self.tools:Dict[str,Dict] = {}
        self.excution_log = []
    def register_tool(self, name:str, func:Callable, description:str):
        '''Tool을 에이전트에 등록합니다.'''
        self.tools[name] = {'func':func, 'description':description}
    def _build_system_prompt(self):
        """등록된 Tool 정보를 포함한 시스템 프롬프트를 생성합니다."""
        tool_desc = "\n".join([f"- {n}: {info['description']}" for n, info in self.tools.items()])
        return f"""당신은 주어진 질문에 정확하게 답하기 위해 도구를 사용하는 AI 에이전트입니다.

사용 가능한 도구:
{tool_desc}

반드시 다음 형식을 따르세요:
Thought: [추론 과정]
Action: [도구이름][입력값]

도구 실행 결과가 Observation으로 주어집니다.
답변이 준비되면:
Thought: [최종 추론]
Action: Finish[최종 답변]

중요 규칙:
- 한 번에 하나의 Action만 수행
- 도구 실행이 실패하면 다른 접근을 시도
- 확실하지 않은 정보는 도구를 통해 확인"""
    
    def _parse_action(self, text):
        """LLM 응답에서 Action과 입력값을 추출합니다."""
        match = re.search(r'Action\s*:\s*(\w+)\[(.+?)\]', text, re.DOTALL)
        if match:
            return match.group(1).strip(), match.group(2).strip()
        return None, None
    
    def _execute_with_retry(self, tool_name, tool_input):
        """재시도 로직이 포함된 Tool 실행 메서드입니다."""
        for attempt in range(self.retry_count + 1):
            try:
                if tool_name in self.tools:
                    result = self.tools[tool_name]["func"](tool_input)
                    return result
                return f"알 수 없는 도구: {tool_name}. 사용 가능: {list(self.tools.keys())}"
            except Exception as e:
                if attempt < self.retry_count:
                    time.sleep(1)
                    continue
                return f"도구 실행 실패 ({attempt + 1}회 시도): {str(e)}"
    
    def run(self, question):
        """질문에 대해 ReAct 루프를 실행합니다."""
        system_prompt = self._build_system_prompt()
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Question: {question}"}
        ]
        self.execution_log = [{"type": "question", "content": question}]
        
        print(f"Question: {question}")
        print("=" * 60)
        
        for step in range(self.max_steps):
            try:
                response = client.chat.completions.create(
                    model=self.model,
                    messages=messages,
                    temperature=0,
                    max_tokens=500
                )
            except Exception as e:
                print(f"API 호출 오류: {e}")
                self.execution_log.append({"type": "error", "content": str(e)})
                break
            
            assistant_msg = response.choices[0].message.content
            messages.append({"role": "assistant", "content": assistant_msg})
            
            print(f"\n--- Step {step + 1} ---")
            print(assistant_msg)
            
            self.execution_log.append({"type": "step", "step": step + 1, "content": assistant_msg})
            
            action_name, action_input = self._parse_action(assistant_msg)
            
            if action_name == "Finish":
                print(f"\n{'=' * 60}")
                print(f"Final Answer: {action_input}")
                self.execution_log.append({"type": "answer", "content": action_input})
                return action_input
            
            if action_name:
                observation = self._execute_with_retry(action_name, action_input)
                obs_msg = f"Observation: {observation}"
                messages.append({"role": "user", "content": obs_msg})
                print(f"\n{obs_msg}")
                self.execution_log.append({"type": "observation", "tool": action_name, "content": observation})
            else:
                error_msg = "형식 오류: Action: 도구이름[입력값] 형식으로 응답해주세요."
                messages.append({"role": "user", "content": error_msg})
                self.execution_log.append({"type": "format_error", "content": assistant_msg})
        
        print("\n최대 단계에 도달했습니다.")
        return None
    
    def get_log(self):
        """실행 로그를 반환합니다."""
        return self.execution_log
    
    def print_summary(self):
        """실행 요약을 출력합니다."""
        print("\n=== 실행 요약 ===")
        for entry in self.execution_log:
            if entry['type'] == 'question':
                print(f"[질문] {entry['content']}")
            elif entry['type'] == 'step':
                print(f"[Step {entry['step']}] LLM 추론")
            elif entry['type'] == 'observation':
                print(f"[관측] Tool={entry['tool']}: {entry['content'][:60]}...")
            elif entry['type'] == 'answer':
                print(f"[답변] {entry['content']}")
            elif entry['type'] == 'error':
                print(f"[오류] {entry['content']}")

# 에이전트 생성 및 도구 등록
agent = RobustReActAgent(model="gpt-4o-mini", max_steps=6, retry_count=2)
agent.register_tool("NewsSearch", fetch_news, "최신 뉴스를 검색합니다. 예: NewsSearch[검색어]")
agent.register_tool("Summarize", summarize_text, "텍스트를 요약합니다. 예: Summarize[요약할 텍스트]")
agent.register_tool("Calculator", 
    lambda x: str(eval(x)) if all(c in '0123456789+-*/.() ' for c in x) else "허용되지 않는 수식", 
    "수학 계산을 수행합니다.")

# 에이전트 실행
result = agent.run("최근 인공지능 관련 뉴스를 검색하고, 주요 내용을 요약해주세요.")
agent.print_summary()

Question: 최근 인공지능 관련 뉴스를 검색하고, 주요 내용을 요약해주세요.

--- Step 1 ---
Thought: 최근 인공지능 관련 뉴스를 검색하여 주요 내용을 파악해야 합니다. 
Action: NewsSearch[인공지능]

Observation: 인공지능관련 최신 뉴스 3건
  1. [전자신문 '미토스 충격' 대응…정부, 2027년 독자 AI 보안체계 구축 - 전자신문  Fri, 29 May 2026 03:06:25 GMT]  2. [v.daum.net AI 사이버위협 '비상'…정부, 민간 보안 총력 대응 - v.daum.net  Fri, 29 May 2026 04:35:38 GMT]  3. [지디넷코리아 스노우플레이크, 나토마 인수…"AI 에이전트 보안 연결·통제 강화" - 지디넷코리아  Fri, 29 May 2026 02:19:54 GMT]

--- Step 2 ---
Thought: 검색 결과를 바탕으로 각 뉴스의 주요 내용을 요약해야 합니다. 
Action: Summarize[미토스 충격' 대응…정부, 2027년 독자 AI 보안체계 구축; AI 사이버위협 '비상'…정부, 민간 보안 총력 대응; 스노우플레이크, 나토마 인수…"AI 에이전트 보안 연결·통제 강화"]

Observation: 요약 : 정부, 2027년 독자 AI 보안체계 구축…민간과 AI 사이버위협 대응 총력. 스노우플레이크는 나토마 인수로 에이전트 보안 강화.

--- Step 3 ---
Thought: 인공지능 관련 뉴스의 주요 내용을 요약한 결과를 바탕으로 최종 답변을 준비합니다. 
Action: Finish[최근 인공지능 관련 뉴스에 따르면, 정부는 2027년까지 독자적인 AI 보안체계를 구축하고 민간과 함께 AI 사이버 위협에 총력 대응할 계획입니다. 또한, 스노우플레이크는 나토마를 인수하여 AI 에이전트의 보안을 강화할 예정입니다.]

Final Answer: 최근 인공지능 관련 뉴스에 따르면, 정부는 2027년까지 독자적인 AI 보안체계를 구축하고 민간과 함께 

## 4. 대화 히스토리를 유지하는 ReAct 에이전트

지금까지 구현한 에이전트는 각 질문을 독립적으로 처리했다. 그러나 실제 대화 시나리오에서는 이전 대화 맥락을 기억하고 참조할 수 있어야 한다.

예를 들어 사용자가 "100 * 25를 계산해줘"라고 물은 후, "그 결과에 500을 더해줘"라고 요청하면, 에이전트는 이전 대화에서 2500이라는 결과를 기억하고 있어야 한다.

### ConversationalReActAgent의 핵심 설계

- **대화 히스토리 누적**: `conversation_history` 리스트에 사용자 질문과 에이전트 답변을 누적
- **맥락 전달**: 매 실행 시 전체 대화 히스토리를 LLM에 전달
- **RobustReActAgent 상속**: 기존의 견고한 에이전트 기능을 그대로 활용

In [15]:
class ConversationReActAgent(RobustReActAgent):
    def __init__(self, **kwargs):  # 패킹
        super().__init__(**kwargs) # 언패킹
        self.conversation_history=[]
        self.systemp_prompt = None
    def run(self,question):
        '''대화 히스토리를 유지하며 질문에 답변합니다.'''
        if self.systemp_prompt is None:
            self.systemp_prompt = self._build_system_prompt()
            self.systemp_prompt += '\n\n추가 규칙:  이전 대화 맥락을 참고하여 답변하세요'
        self.conversation_history.append({'role':'user', 'content':f'Question : {question}'})
        messages = [{'role':'system','content':self.systemp_prompt}] + self.conversation_history.copy()        
        self.execution_log = [{'type':'question','content':question}]
        print(f"\nQuestion: {question}")
        print("=" * 60)
        
        for step in range(self.max_steps):
            response = client.chat.completions.create(
                model=self.model, messages=messages, temperature=0, max_completion_tokens=500
            )
            
            assistant_msg = response.choices[0].message.content
            messages.append({"role": "assistant", "content": assistant_msg})
            
            print(f"\n--- Step {step + 1} ---")
            print(assistant_msg)
            self.execution_log.append({"type": "step", "step": step+1, "content": assistant_msg})
            
            action_name, action_input = self._parse_action(assistant_msg)
            
            if action_name == "Finish":
                self.conversation_history.append({"role": "assistant", "content": action_input})
                print(f"\nFinal Answer: {action_input}")
                return action_input
            
            if action_name:
                observation = self._execute_with_retry(action_name, action_input)
                obs_msg = f"Observation: {observation}"
                messages.append({"role": "user", "content": obs_msg})
                print(f"\n{obs_msg}")
            else:
                messages.append({"role": "user", "content": "Action: 도구이름[입력값] 형식으로 응답해주세요."})
        
        return None


In [16]:
print(">>>> 첫번째 질문")
agent.run('100*25의 결과를 알려주세요')
print("\n\n>>>> 두번째 질문")
agent.run('방금계산한 결과에 500을 더하면 얼마인가요?')

>>>> 첫번째 질문
Question: 100*25의 결과를 알려주세요

--- Step 1 ---
Thought: 100과 25를 곱한 결과를 계산해야 합니다.
Action: Calculator[100*25]

Observation: 2500

--- Step 2 ---
Thought: 100과 25를 곱한 결과는 2500입니다.
Action: Finish[100 * 25의 결과는 2500입니다.]

Final Answer: 100 * 25의 결과는 2500입니다.


>>>> 두번째 질문
Question: 방금계산한 결과에 500을 더하면 얼마인가요?

--- Step 1 ---
Thought: 방금 계산한 결과가 무엇인지 알 수 없으므로, 이전 계산 결과를 확인해야 합니다. 하지만 이전 계산 결과가 없으므로, 500을 더할 수 있는 수치를 알 수 없습니다. 따라서, 추가적인 정보가 필요합니다. 
Action: Finish[이전 계산 결과를 제공해 주시면, 그 결과에 500을 더한 값을 계산해 드리겠습니다.]

Final Answer: 이전 계산 결과를 제공해 주시면, 그 결과에 500을 더한 값을 계산해 드리겠습니다.


'이전 계산 결과를 제공해 주시면, 그 결과에 500을 더한 값을 계산해 드리겠습니다.'

In [17]:
conv_agent = ConversationReActAgent()
conv_agent.register_tool("NewsSearch", fetch_news, "최신 뉴스를 검색합니다. 예: NewsSearch[검색어]")
conv_agent.register_tool("Summarize", summarize_text, "텍스트를 요약합니다. 예: Summarize[요약할 텍스트]")
conv_agent.register_tool("Calculator", 
    lambda x: str(eval(x)) if all(c in '0123456789+-*/.() ' for c in x) else "허용되지 않는 수식", 
    "수학 계산을 수행합니다.")

In [18]:
print(">>>> 첫번째 질문")
conv_agent.run('100*25의 결과를 알려주세요')
print("\n\n>>>> 두번째 질문")
conv_agent.run('방금계산한 결과에 500을 더하면 얼마인가요?')

>>>> 첫번째 질문

Question: 100*25의 결과를 알려주세요

--- Step 1 ---
Thought: 100*25를 계산하면 됩니다.  
Action: Calculator[100*25]

Observation: 2500

--- Step 2 ---
Thought: 계산 결과를 그대로 답하면 됩니다.  
Action: Finish[100*25 = 2500]

Final Answer: 100*25 = 2500


>>>> 두번째 질문

Question: 방금계산한 결과에 500을 더하면 얼마인가요?

--- Step 1 ---
Thought: 이전 계산 결과(2500)에 500을 더하면 된다. 계산을 위해 도구가 필요 없지만 정확히 산술을 수행한다.
Action: Finish[2500 + 500 = 3000]

Final Answer: 2500 + 500 = 3000


'2500 + 500 = 3000'

In [19]:
print(">>>> 첫번째 질문")
conv_agent.run('100*25의 결과를 알려주세요')

conv_agent.conversation_history

>>>> 첫번째 질문

Question: 100*25의 결과를 알려주세요

--- Step 1 ---
Thought: [100*25의 곱을 계산하면 된다.]
Action: Calculator[100*25]

Observation: 2500

--- Step 2 ---
Thought: [곱셈 결과는 2500이므로 그대로 답한다.]
Action: Finish[100*25의 결과는 2500입니다.]

Final Answer: 100*25의 결과는 2500입니다.


[{'role': 'user', 'content': 'Question : 100*25의 결과를 알려주세요'},
 {'role': 'assistant', 'content': '100*25 = 2500'},
 {'role': 'user', 'content': 'Question : 방금계산한 결과에 500을 더하면 얼마인가요?'},
 {'role': 'assistant', 'content': '2500 + 500 = 3000'},
 {'role': 'user', 'content': 'Question : 100*25의 결과를 알려주세요'},
 {'role': 'assistant', 'content': '100*25의 결과는 2500입니다.'}]

## 5. 정리 및 핵심 요약

### 커스텀 Tool 설계 원칙

| 원칙 | 설명 | 적용 사례 |
|------|------|----------|
| 단일 책임 | 하나의 Tool은 하나의 기능만 수행 | 뉴스 검색 Tool, 요약 Tool 분리 |
| 명확한 설명문 | LLM이 올바르게 도구를 선택하도록 유도 | 기능, 입력 형식, 예시 포함 |
| 표준화된 인터페이스 | 문자열 입력/출력 형식 일관성 유지 | 모든 Tool이 str -> str 형태 |
| 에러 처리 | 예외 상황에서도 의미 있는 결과 반환 | try/except, 타임아웃, 재시도 |

### 구현한 컴포넌트

1. **fetch_news Tool**: Google News RSS를 활용한 실시간 뉴스 검색 도구
2. **summarize_text Tool**: OpenAI API 기반의 텍스트 요약 도구
3. **RobustReActAgent**: 재시도 로직, 실행 로그, 형식 오류 복구가 포함된 견고한 에이전트
4. **ConversationalReActAgent**: 대화 히스토리를 유지하여 맥락을 이해하는 대화형 에이전트

### 프로덕션 레벨 ReAct 워크플로우 구축 (LangGraph)
- LangGraph를 활용하여 상태(State) 기반의 에이전트 워크플로우를 설계한다.
- 순환(Cyclic) 그래프 구조를 통해 모델이 원하는 결과를 낼 때까지 스스로 도구를 호출하고 검증하는 로직을 구현한다.

### 1. 상태(State) 및 도구 정의
LangGraph는 각 노드(Node) 간에 전달되는 상태(State)를 정의해야 합니다. 에이전트의 상태는 주고받은 메시지들의 목록입니다.

In [20]:
import os
import operator
from typing import TypedDict, Annotated, Sequence
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage,ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

In [21]:
# 1. 상태객체(state object) 정의
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

# 2. 도구 정의
@tool
def get_current_time(loaction:str)->str:
    '''특정 지역의 현재 시간을 반환합니다.'''
    import datetime
    return f"{loaction}의 현재 시간은 {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} 입니다"

tools = [get_current_time]
# langgraph.prebuilt 의 ToolNode 를 사용하면 실행노드를 쉽게 만들수 있음
tool_node = ToolNode(tools)
print('상태 정의 및 도구 초기화 완료')

상태 정의 및 도구 초기화 완료


### 2. 에이전트 노드 및 라우팅 함수 구현
모델을 호출하는 노드(Node)와, 모델의 응답에 따라 도구를 실행할지 종료할지 결정하는 엣지(Edge) 로직을 만듭니다.

In [22]:
# 1. 모델과 도구 바인딩(Function calling)
model = ChatOpenAI(api_key=os.getenv('OPENAI_API_KEY'),model='gpt-5.4-nano',temperature=0)
model_with_tools = model.bind_tools(tools)

# 2. 에이전트 노드 함수 : 상태를 받아 모델을 호출하고 상태를 업데이트
def call_model(state:AgentState):
    messages = state['messages']
    response = model_with_tools.invoke(messages)
    return {'messages':[response]}

# 3.라우팅 로직 : 모델이 도구 호출(tool_calls)을 요청했는지 확인
def should_continue(state:AgentState):
    messages = state['messages']
    last_message = messages[-1]
    # 마지막 메세지에 도구 호출 정보가 있다면 도구 노드로 이동
    if last_message.tool_calls:
        return 'continue'
    # 도구 호출이 없다면(최종답변을 도출했다면) 종료
    return 'end'

print('에이전트 노드와 라우터 준비')

에이전트 노드와 라우터 준비


# [5교시]

### 3. StateGraph 워크플로우 조립 및 컴파일
노드와 엣지를 연결하여 전체 ReAct 사이클(추론 -> 도구실행 -> 관찰 -> 추론...)을 완성합니다.

In [23]:
# 워크플로우 그래프 생성  # 노드(node) 엣지(edge) 흐름(flow)
workflow = StateGraph(AgentState)
# 노드 등록
workflow.add_node('agent', call_model)  # 추론 노드
workflow.add_node('action',tool_node)  # 행동(도구) 노드

# 시작점을 agent로 설정
workflow.set_entry_point('agent')

# workflow.add_edge('a','b')  # 무조건 a->b
# 조건부 엣지 등록 : agent의 결과에 따라서 분기
workflow.add_conditional_edges(
    'agent',   # 현재노드 이름
    should_continue,  #라우터 : 분기판단
    {
        'continue':'action',   # 라우터가 continue를 반환하면 action 노드로 이동
        'end':END   # 종료
    }
)

# 일반 엣지 등록 : action(도구 실행)이 끝나면 무조건 agent(추론)으로 다시 돌아가서 관찰결과 전달
workflow.add_edge('action','agent')

# 그래프 컴파일
app = workflow.compile()

print('ReAct 워크플로우 컴파일 완료')

ReAct 워크플로우 컴파일 완료


### 4. 워크플로우 실행 및 스트리밍
구성된 그래프에 초기 상태(사용자 질문)를 입력하고 실행 흐름을 단계별로 추적해봅니다.

In [24]:
inputs = {'messages' : [HumanMessage(content='서울의 현재  시간은 언제인가요?')]}
print(' == LangGraph ReAct 실행 흐름 ==')

for output in app.stream(inputs):
    for node_name, state_update in output.items():
        print(f"\n[노드 실행 완료]: {node_name}")
        
        # 업데이트된 마지막 메시지 확인
        latest_message = state_update['messages'][-1]
        
        if isinstance(latest_message, AIMessage):
            if hasattr(latest_message, "tool_calls") and latest_message.tool_calls:
                tool_call = latest_message.tool_calls[0]
                print(f" -> Thought: 도구 '{tool_call['name']}' 호출 결정")
            else:
                print(f" -> Final Answer: {latest_message.content}")
        elif isinstance(latest_message, ToolMessage):
            print(f" -> Observation: {latest_message.content}")

print("\n=== 실행 종료 ===")


 == LangGraph ReAct 실행 흐름 ==

[노드 실행 완료]: agent
 -> Thought: 도구 'get_current_time' 호출 결정

[노드 실행 완료]: action
 -> Observation: 서울의 현재 시간은 2026-05-29 14:29:24 입니다

[노드 실행 완료]: agent
 -> Final Answer: 서울의 현재 시간은 **2026-05-29 14:29:24** 입니다.

=== 실행 종료 ===


# [6교시]

## 학습 목표

1. **이전 ReAct 과정과 RAG의 연결점**을 이해하고, 프롬프팅 기법의 발전 흐름 속에서 RAG의 위치를 설명할 수 있다.
2. **LLM의 본질적 한계**(지식 컷오프, 환각, 도메인 전문성 부족)를 구체적으로 인식한다.
3. **RAG(Retrieval-Augmented Generation)** 의 핵심 아이디어와 아키텍처를 이해한다.
4. **ReAct와 RAG의 공통점과 차이점** 을 비교 분석할 수 있다.
5. RAG 파이프라인의 3단계(Indexing, Retrieval, Generation) 구조를 파악한다.

## 1. 이전 과정 회고와 RAG로의 전환

이전 ReAct 과정에서 우리는 **Thought-Action-Observation(TAO) 루프** 를 학습하고, LLM이 외부 도구(Search, Calculator 등)를 호출하여 정보를 수집하는 **에이전트 워크플로우** 를 구축했습니다.

ReAct의 핵심 통찰은 다음과 같았습니다:
- LLM은 **내부 지식만으로는 한계** 가 있다
- **외부 도구를 통해 실시간 정보를 수집** 하면 환각을 줄일 수 있다
- 추론(Reasoning)과 행동(Acting)을 **교차 반복** 하여 정확도를 높인다

이번 RAG 과정에서는 이 개념을 **더 체계적이고 구조화된 방향** 으로 확장합니다.

```
프롬프팅 기법의 발전:
Zero-shot → Few-shot → CoT → ReAct → RAG
(단순 질의)  (예시 제공)  (추론 명시)  (추론+행동)  (검색+생성)

ReAct의 Search Tool  ──────→  RAG의 Retrieval 단계
(매번 실시간 검색)            (사전 구축된 지식 베이스에서 검색)
```

**핵심 전환점**: ReAct에서는 Search API를 통해 매번 외부 검색을 수행했지만, RAG에서는 **사전에 문서를 벡터화하여 저장** 해 두고, 질문이 들어오면 **벡터 유사도 기반으로 관련 문서를 빠르게 검색** 합니다. 이를 통해 검색 속도, 정확도, 비용 면에서 큰 이점을 얻습니다.

## 2. LLM의 본질적 한계

대규모 언어 모델(LLM)은 방대한 텍스트 데이터를 학습하여 놀라운 언어 생성 능력을 보여주지만, 본질적으로 다음과 같은 한계를 가지고 있습니다.

### 2.1 지식 컷오프(Knowledge Cutoff)

LLM은 학습 데이터의 시점 이후에 발생한 사건이나 정보를 알지 못합니다. 예를 들어 2024년까지의 데이터로 학습된 모델은 2025년의 사건에 대해 정확한 답변을 제공할 수 없습니다.

### 2.2 환각(Hallucination)

LLM은 학습 데이터에 존재하지 않는 정보를 마치 사실인 것처럼 생성할 수 있습니다. 이는 모델이 확률적으로 그럴듯한 텍스트를 생성하는 방식에서 기인합니다.

### 2.3 도메인 전문성 부족

범용 LLM은 특정 기업의 내부 문서, 전문 분야의 최신 연구, 비공개 데이터베이스의 정보에 접근할 수 없습니다. 도메인 특화된 질문에 대해 일반적이거나 부정확한 답변을 제공할 가능성이 높습니다.

In [25]:
print('''
============================================================
[한계 1] 지식 컷오프(Knowledge Cutoff)
============================================================
  질문: 2025년 노벨 물리학상 수상자는 누구인가요?
  LLM 예상 응답: 학습 데이터 이후의 정보이므로 정확한 답변이 어렵습니다.
  문제점: 학습 데이터의 시점(cutoff) 이후 정보에 접근 불가
  RAG 해결책: 최신 문서를 벡터 DB에 저장하여 검색 후 답변 생성

============================================================
[한계 2] 환각(Hallucination)
============================================================
  질문: XYZ-3000 프로토콜의 주요 특징은 무엇인가요?
  LLM 예상 응답: XYZ-3000은 분산 시스템에서 사용되는... (존재하지 않는 정보를 생성)
  문제점: 존재하지 않는 정보를 사실처럼 생성
  RAG 해결책: 검증된 문서에서만 정보를 검색하여 근거 기반 답변 생성

============================================================
[한계 3] 도메인 전문성 부족
============================================================
  질문: 우리 회사의 내부 보안 정책 중 비밀번호 규칙은?
  LLM 예상 응답: 일반적인 보안 정책을 설명하지만 특정 회사 정보는 없음
  문제점: 비공개/특정 도메인 데이터에 대한 지식 없음
  RAG 해결책: 회사 내부 문서를 벡터 DB에 인덱싱하여 도메인 특화 답변
''')


[한계 1] 지식 컷오프(Knowledge Cutoff)
  질문: 2025년 노벨 물리학상 수상자는 누구인가요?
  LLM 예상 응답: 학습 데이터 이후의 정보이므로 정확한 답변이 어렵습니다.
  문제점: 학습 데이터의 시점(cutoff) 이후 정보에 접근 불가
  RAG 해결책: 최신 문서를 벡터 DB에 저장하여 검색 후 답변 생성

[한계 2] 환각(Hallucination)
  질문: XYZ-3000 프로토콜의 주요 특징은 무엇인가요?
  LLM 예상 응답: XYZ-3000은 분산 시스템에서 사용되는... (존재하지 않는 정보를 생성)
  문제점: 존재하지 않는 정보를 사실처럼 생성
  RAG 해결책: 검증된 문서에서만 정보를 검색하여 근거 기반 답변 생성

[한계 3] 도메인 전문성 부족
  질문: 우리 회사의 내부 보안 정책 중 비밀번호 규칙은?
  LLM 예상 응답: 일반적인 보안 정책을 설명하지만 특정 회사 정보는 없음
  문제점: 비공개/특정 도메인 데이터에 대한 지식 없음
  RAG 해결책: 회사 내부 문서를 벡터 DB에 인덱싱하여 도메인 특화 답변



## 3. RAG(Retrieval-Augmented Generation)의 핵심 아이디어

### 3.1 RAG의 정의

**RAG(Retrieval-Augmented Generation, 검색 증강 생성)** 은 2020년 Facebook AI Research(현 Meta AI)의 Lewis et al.이 발표한 기술로, **정보 검색(Retrieval)** 과 **텍스트 생성(Generation)** 을 결합한 하이브리드 접근법입니다.

핵심 원리는 매우 단순합니다:

```
사용자 질문 → 관련 문서 검색(Retrieval) → 검색된 문서를 컨텍스트로 LLM에 전달 → 답변 생성(Generation)
```

### 3.2 RAG의 3단계 아키텍처

```
┌─────────────────────────────────────────────────────────────┐
│                    RAG 파이프라인                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  [1단계: Indexing (인덱싱)]                                  │
│  문서 수집 → 전처리 → 청킹 → 임베딩 → 벡터 DB 저장          │
│                                                             │
│  [2단계: Retrieval (검색)]                                   │
│  사용자 질문 → 질문 임베딩 → 벡터 유사도 검색 → 관련 문서 반환│
│                                                             │
│  [3단계: Generation (생성)]                                  │
│  프롬프트 구성(질문 + 검색 문서) → LLM 호출 → 답변 생성       │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 3.3 RAG가 LLM 한계를 해결하는 방식

| LLM 한계 | RAG의 해결 방식 |
|---------|---------------|
| 지식 컷오프 | 최신 문서를 벡터 DB에 지속 업데이트 |
| 환각 | 검증된 문서 기반으로만 답변 생성 |
| 도메인 전문성 | 도메인 특화 문서를 인덱싱하여 전문 지식 제공 |

### ReAct vs Rag 공통점과 핵심 차이점

In [26]:
print('''
비교 항목              ReAct                        RAG                         
--------------------------------------------------------------------------
핵심 원리              추론 + 행동 교차 반복                검색 + 생성 결합                  
외부 지식 활용           Tool(Search API 등) 호출        벡터 DB에서 문서 검색               
검색 방식              도구 기반 실시간 검색                 벡터 유사도 기반 검색                
지식 저장              없음 (매번 검색)                   사전 구축된 벡터 DB                
실시간성               높음                           중간 (업데이트 주기 의존)             
구현 복잡도             중간                           중간~높음                       
주요 용도              동적 질의응답, 다단계 추론              도메인 특화 QA, 문서 기반 답변         
환각 감소              높음 (외부 검증)                   높음 (근거 문서 제공)               
응답 속도              느림 (다단계)                     빠름 (단일 검색+생성)               

==========================================================================
[공통점]
  - 모두 LLM의 내부 지식 한계를 외부 정보로 보완
  - 환각(hallucination)을 줄이기 위한 접근법
  - 외부 소스에서 정보를 수집하여 답변의 사실성 향상

[핵심 차이점]
  - ReAct: 실시간 도구 호출 → 동적이지만 느림
  - RAG: 사전 구축된 지식 베이스 → 빠르지만 업데이트 필요
      ''')


비교 항목              ReAct                        RAG                         
--------------------------------------------------------------------------
핵심 원리              추론 + 행동 교차 반복                검색 + 생성 결합                  
외부 지식 활용           Tool(Search API 등) 호출        벡터 DB에서 문서 검색               
검색 방식              도구 기반 실시간 검색                 벡터 유사도 기반 검색                
지식 저장              없음 (매번 검색)                   사전 구축된 벡터 DB                
실시간성               높음                           중간 (업데이트 주기 의존)             
구현 복잡도             중간                           중간~높음                       
주요 용도              동적 질의응답, 다단계 추론              도메인 특화 QA, 문서 기반 답변         
환각 감소              높음 (외부 검증)                   높음 (근거 문서 제공)               
응답 속도              느림 (다단계)                     빠름 (단일 검색+생성)               

[공통점]
  - 모두 LLM의 내부 지식 한계를 외부 정보로 보완
  - 환각(hallucination)을 줄이기 위한 접근법
  - 외부 소스에서 정보를 수집하여 답변의 사실성 향상

[핵심 차이점]
  - ReAct: 실시간 도구 호출 → 동적이지만 느림
  - RAG

## 1. RAG에서 텍스트 전처리의 중요성

RAG 파이프라인의 첫 번째 단계는 **Indexing(인덱싱)** 이었습니다. 이 단계에서 문서를 검색 가능한 형태로 변환하는 과정은 다음과 같습니다:

```
원본 문서 → 로딩 → 클리닝(전처리) → 청킹(분할) → 임베딩(벡터화) → 벡터 DB 저장
```

이 중 **청킹(Chunking)** 은 RAG 성능에 가장 큰 영향을 미치는 단계 중 하나입니다.

### 왜 청킹이 중요한가?

| 청크가 너무 작을 때 | 청크가 너무 클 때 |
|:--:|:--:|
| 문맥 정보가 손실됨 | 관련 없는 내용이 포함됨 |
| 의미가 불완전한 조각 | 검색 정확도 하락 |
| 검색 결과는 많지만 품질 낮음 | LLM 토큰 한도 초과 위험 |

따라서 **적절한 청크 크기와 분할 전략을 선택** 하는 것이 RAG 시스템의 핵심 설계 결정입니다.

In [27]:
import re

# RAG 관련 한국어 기술 문서
sample_document = """
검색 증강 생성(RAG) 기술 개요

1. RAG의 정의와 배경

RAG(Retrieval-Augmented Generation)는 2020년 Meta AI 연구팀이 발표한 혁신적인 기술이다. 이 기술은 대규모 언어 모델(LLM)의 텍스트 생성 능력과 정보 검색 시스템을 결합한 하이브리드 접근법이다. 기존의 LLM은 학습 데이터에만 의존하여 답변을 생성했기 때문에, 학습 이후의 정보나 특정 도메인의 전문 지식에 대해서는 정확한 답변을 제공하기 어려웠다. RAG는 이러한 한계를 극복하기 위해, 질문이 주어지면 먼저 외부 지식 소스에서 관련 문서를 검색하고, 검색된 문서를 컨텍스트로 활용하여 답변을 생성한다.

2. 벡터 데이터베이스의 역할

벡터 데이터베이스는 RAG 시스템의 핵심 인프라이다. 텍스트 데이터를 고차원 벡터(임베딩)로 변환하여 저장하고, 코사인 유사도를 기반으로 의미적으로 유사한 문서를 빠르게 검색할 수 있게 해준다. 대표적인 벡터 데이터베이스로는 ChromaDB, FAISS, Pinecone, Weaviate, Milvus 등이 있다. 각 데이터베이스는 인덱싱 방식, 검색 속도, 확장성 면에서 고유한 특성을 가진다.

3. 임베딩 모델과 의미적 검색

임베딩(Embedding)은 텍스트를 고차원 수치 벡터로 변환하는 과정이다. 의미적으로 유사한 텍스트는 벡터 공간에서 가까이 위치하게 된다. OpenAI의 text-embedding-3-small 모델은 1536차원의 벡터를 생성하며, 다국어를 지원한다. 임베딩을 통해 키워드 매칭이 아닌 의미적 유사성 기반의 검색이 가능해진다.

4. 청킹 전략의 중요성

청킹(Chunking)은 긴 문서를 적절한 크기의 작은 단위로 분할하는 과정이다. 청크의 크기가 너무 작으면 문맥 정보가 손실되고, 너무 크면 검색 정확도가 떨어진다. 주요 청킹 전략으로는 고정 크기 청킹, 문장 기반 청킹, 재귀적 청킹 등이 있다. 각 전략은 문서의 특성과 사용 목적에 따라 적절히 선택해야 한다.

5. 고급 RAG 패턴

기본 RAG의 한계를 극복하기 위한 다양한 고급 패턴이 연구되고 있다. Query Rewriting은 사용자의 질문을 검색에 최적화된 형태로 변환하는 기법이다. HyDE(Hypothetical Document Embedding)는 질문에 대한 가상 답변을 생성하여 검색에 활용한다. Re-ranking은 초기 검색 결과를 재정렬하여 관련성을 높인다. Multi-Query는 하나의 질문에서 여러 검색 질의를 생성하여 검색 범위를 확장한다.

6. 지식 그래프와 Hybrid RAG

지식 그래프(Knowledge Graph)는 엔티티와 관계를 그래프 구조로 표현하는 방식이다. 벡터 검색이 의미적 유사성에 강한 반면, 지식 그래프는 구조적 관계 추론에 강하다. Hybrid RAG는 벡터 검색과 그래프 검색을 결합하여 두 가지 장점을 모두 활용하는 접근법이다. 이를 통해 의미적 유사성과 구조적 관계를 동시에 고려한 검색이 가능해진다.
"""

# 텍스트 클리닝 함수
def clean_text(text):
    """텍스트 전처리: 불필요한 공백과 특수문자 정리"""
    text = text.strip()  # 앞뒤 공백 제거
    text = re.sub(r'\n{3,}', '\n\n', text)  # 연속 줄바꿈 정리
    text = re.sub(r' {2,}', ' ', text)  # 연속 공백 정리
    return text

cleaned_text = clean_text(sample_document)

print(f"원본 문서 길이: {len(sample_document)}자")
print(f"클리닝 후 길이: {len(cleaned_text)}자")
print(f"줄 수: {len(cleaned_text.splitlines())}줄")
print()
print("[문서 미리보기 (처음 200자)]")
print(cleaned_text[:200] + "...")

원본 문서 길이: 1467자
클리닝 후 길이: 1465자
줄 수: 25줄

[문서 미리보기 (처음 200자)]
검색 증강 생성(RAG) 기술 개요

1. RAG의 정의와 배경

RAG(Retrieval-Augmented Generation)는 2020년 Meta AI 연구팀이 발표한 혁신적인 기술이다. 이 기술은 대규모 언어 모델(LLM)의 텍스트 생성 능력과 정보 검색 시스템을 결합한 하이브리드 접근법이다. 기존의 LLM은 학습 데이터에만 의존하여 답변을 생성했기...


## 3. 고정 크기 청킹 (Fixed-Size Chunking)

가장 단순한 청킹 방식으로, **문자 수 기준** 으로 일정한 크기의 청크를 생성합니다. 오버랩(overlap)을 적용하면 청크 간의 정보 손실을 줄일 수 있습니다.

In [28]:
# 고정 크기 청킹 구현
# 문자 수 기준으로 일정한 크기로 텍스트를 분할합니다.

def fixed_size_chunk(text, chunk_size=200, overlap=50):
    """고정 크기 청킹: chunk_size 단위로 분할, overlap만큼 겹침"""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append({
            'text': chunk,
            'start': start,
            'end': min(end, len(text)),
            'length': len(chunk)
        })
        start = end - overlap  # 오버랩 적용
        if start >= len(text):
            break
    return chunks

# 오버랩 없는 청킹
chunks_no_overlap = fixed_size_chunk(cleaned_text, chunk_size=200, overlap=0)

# 오버랩 있는 청킹
chunks_with_overlap = fixed_size_chunk(cleaned_text, chunk_size=200, overlap=50)

print("=" * 60)
print("[고정 크기 청킹 - 오버랩 없음 (chunk_size=200)]")
print("=" * 60)
print(f"총 청크 수: {len(chunks_no_overlap)}개")
for i, chunk in enumerate(chunks_no_overlap[:3]):
    print(f"\n--- 청크 {i+1} (위치: {chunk['start']}~{chunk['end']}, 길이: {chunk['length']}자) ---")
    print(chunk['text'][:100] + "..." if len(chunk['text']) > 100 else chunk['text'])

print(f"\n\n{'=' * 60}")
print("[고정 크기 청킹 - 오버랩 있음 (chunk_size=200, overlap=50)]")
print("=" * 60)
print(f"총 청크 수: {len(chunks_with_overlap)}개")
for i, chunk in enumerate(chunks_with_overlap[:3]):
    print(f"\n--- 청크 {i+1} (위치: {chunk['start']}~{chunk['end']}, 길이: {chunk['length']}자) ---")
    print(chunk['text'][:100] + "..." if len(chunk['text']) > 100 else chunk['text'])

[고정 크기 청킹 - 오버랩 없음 (chunk_size=200)]
총 청크 수: 8개

--- 청크 1 (위치: 0~200, 길이: 200자) ---
검색 증강 생성(RAG) 기술 개요

1. RAG의 정의와 배경

RAG(Retrieval-Augmented Generation)는 2020년 Meta AI 연구팀이 발표한 혁신적...

--- 청크 2 (위치: 200~400, 길이: 200자) ---
 때문에, 학습 이후의 정보나 특정 도메인의 전문 지식에 대해서는 정확한 답변을 제공하기 어려웠다. RAG는 이러한 한계를 극복하기 위해, 질문이 주어지면 먼저 외부 지식 소스에서...

--- 청크 3 (위치: 400~600, 길이: 200자) ---
 고차원 벡터(임베딩)로 변환하여 저장하고, 코사인 유사도를 기반으로 의미적으로 유사한 문서를 빠르게 검색할 수 있게 해준다. 대표적인 벡터 데이터베이스로는 ChromaDB, FA...


[고정 크기 청킹 - 오버랩 있음 (chunk_size=200, overlap=50)]
총 청크 수: 10개

--- 청크 1 (위치: 0~200, 길이: 200자) ---
검색 증강 생성(RAG) 기술 개요

1. RAG의 정의와 배경

RAG(Retrieval-Augmented Generation)는 2020년 Meta AI 연구팀이 발표한 혁신적...

--- 청크 2 (위치: 150~350, 길이: 200자) ---
을 결합한 하이브리드 접근법이다. 기존의 LLM은 학습 데이터에만 의존하여 답변을 생성했기 때문에, 학습 이후의 정보나 특정 도메인의 전문 지식에 대해서는 정확한 답변을 제공하기 ...

--- 청크 3 (위치: 300~500, 길이: 200자) ---
 관련 문서를 검색하고, 검색된 문서를 컨텍스트로 활용하여 답변을 생성한다.

2. 벡터 데이터베이스의 역할

벡터 데이터베이스는 RAG 시스템의 핵심 인프라이다. 텍스트 데이터를...


In [29]:
sample = '동해물과 백두산이 마르고 닳도록 하느님이 보우하사 우리나라 만세'
fixed_size_chunk(sample,chunk_size=15,overlap=0)

[{'text': '동해물과 백두산이 마르고 닳', 'start': 0, 'end': 15, 'length': 15},
 {'text': '도록 하느님이 보우하사 우리', 'start': 15, 'end': 30, 'length': 15},
 {'text': '나라 만세', 'start': 30, 'end': 35, 'length': 5}]

## 4. 문장 기반 청킹 (Sentence-Based Chunking)

문장 단위로 텍스트를 분할한 후, 지정된 수의 문장을 하나의 청크로 그룹화하는 방식입니다. 문장의 의미적 완결성을 유지하면서 분할할 수 있습니다.

## 5. 재귀적 청킹 (Recursive Chunking)

LangChain에서 가장 많이 사용되는 `RecursiveCharacterTextSplitter`의 원리를 직접 구현합니다. 구분자 계층(`\n\n` → `\n` → `. ` → ` `)을 순차적으로 적용하여 **의미적 단위를 최대한 보존**하면서 분할합니다.


세 가지 전략의 비교 결과를 분석한다.

| 전략 | 특징 | 적합한 상황 |
|------|------|----------|
| 고정 크기 | 균일한 청크 크기, 구현 간단 | 구조가 없는 원시 텍스트, 빠른 프로토타이핑 |
| 문장 기반 | 의미적 완결성 보장, 크기 불균일 | 문장이 명확히 구분된 문서, 정확한 검색 필요 시 |
| 재귀적 | 의미 단위 보존, 가장 유연 | 구조화된 문서, 실무 RAG 시스템 |

**실무 권장**: 대부분의 RAG 시스템에서는 **재귀적 청킹** 이 기본 선택이다. LangChain의 `RecursiveCharacterTextSplitter`가 이 전략을 사용한다.

## 1. 텍스트 임베딩의 개념

청킹을 통해 문서를 작은 단위로 분할하는 방법을 학습했습니다. 이제 이 청크들을 **검색 가능한 형태** 로 만들어야 합니다.

### 임베딩이란?

**임베딩(Embedding)** 은 텍스트를 **고차원 수치 벡터** 로 변환하는 과정입니다. 변환된 벡터는 텍스트의 **의미적 정보** 를 수치로 인코딩합니다.

```
텍스트                          벡터 (고차원)
──────────────────────────────────────────────────
"인공지능은 미래 기술이다"  →  [0.023, -0.015, 0.078, ..., 0.041]  (1536차원)
"AI는 혁신적 기술이다"     →  [0.021, -0.012, 0.075, ..., 0.039]  (유사한 벡터)
"오늘 날씨가 좋다"         →  [-0.045, 0.032, -0.018, ..., 0.063] (다른 벡터)
```

### 핵심 특성

- **의미적으로 유사한 텍스트** 는 벡터 공간에서 **가까이** 위치한다
- **의미적으로 다른 텍스트** 는 벡터 공간에서 **멀리** 위치한다
- 이를 통해 **키워드 매칭이 아닌 의미 기반 검색** 이 가능해진다

## 2. OpenAI Embedding API 활용

OpenAI의 `text-embedding-3-small` 모델을 사용하여 텍스트를 벡터로 변환합니다.

In [31]:
import os
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def get_embedding(text, model = 'text-embedding-3-small'):
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

# 다양한 주제의 텍스트로 임베딩 테스트
texts = [
    "인공지능은 인간의 학습 능력을 컴퓨터로 구현한 기술이다.",
    "머신러닝은 데이터로부터 패턴을 학습하는 AI의 한 분야이다.",
    "오늘 서울의 날씨는 맑고 기온은 25도이다.",
    "파이썬은 데이터 과학에 널리 사용되는 프로그래밍 언어이다.",
]

for text in texts:
    print(f'\n테스트 : {text}')
    print(f'벡터차원 : {len(get_embedding(text))}')
    print(f'벡터 앞 5개 값을 : {get_embedding(text)[:5]}')





테스트 : 인공지능은 인간의 학습 능력을 컴퓨터로 구현한 기술이다.
벡터차원 : 1536
벡터 앞 5개 값을 : [-0.0260009765625, 0.0192413330078125, -0.03167724609375, 0.01084136962890625, 0.0200042724609375]

테스트 : 머신러닝은 데이터로부터 패턴을 학습하는 AI의 한 분야이다.
벡터차원 : 1536
벡터 앞 5개 값을 : [-0.03070068359375, 0.0027408599853515625, 0.0280303955078125, 0.01346588134765625, 0.069580078125]

테스트 : 오늘 서울의 날씨는 맑고 기온은 25도이다.
벡터차원 : 1536
벡터 앞 5개 값을 : [0.0180206298828125, -0.01078033447265625, -0.06939697265625, -0.03668212890625, 0.023529052734375]

테스트 : 파이썬은 데이터 과학에 널리 사용되는 프로그래밍 언어이다.
벡터차원 : 1536
벡터 앞 5개 값을 : [0.002017974853515625, 0.06939697265625, -0.01092529296875, -0.0006690025329589844, 0.0787353515625]


## 3. 코사인 유사도와 벡터 검색

두 벡터 간의 **유사도**를 측정하는 대표적인 방법이 **코사인 유사도(Cosine Similarity)** 입니다. 값이 1에 가까울수록 유사하고, 0에 가까울수록 무관합니다.

In [32]:
embeddings = [ get_embedding(t) for t in texts]
def cosine_similarity(vec_a, vec_b):
    a = np.array(vec_a)
    b = np.array(vec_b)
    return float(np.dot(a,b)/(np.linalg.norm(a) * np.linalg.norm(b)))

# 헤더 출력
print(f"{'':>8}", end="")
for j in range(len(texts)):
    print(f"텍스트{j+1:>2}", end="  ")
print()

for i in range(len(texts)):
    print(f"텍스트{i+1}", end=" ")
    for j in range(len(texts)):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"  {sim:.4f}", end="")
    print()

print()
print("[텍스트 목록]")
for i, t in enumerate(texts):
    print(f"  텍스트{i+1}: {t}")

print()
print("[유사도 해석]")
sim_12 = cosine_similarity(embeddings[0], embeddings[1])
sim_13 = cosine_similarity(embeddings[0], embeddings[2])

print(f"  AI-머신러닝 유사도: {sim_12:.4f}")
print(f"  AI-날씨 유사도:     {sim_13:.4f}")

        텍스트 1  텍스트 2  텍스트 3  텍스트 4  
텍스트1   1.0000  0.4824  0.0036  0.2905
텍스트2   0.4824  1.0000  0.0395  0.3349
텍스트3   0.0036  0.0395  1.0000  0.0920
텍스트4   0.2905  0.3349  0.0920  1.0000

[텍스트 목록]
  텍스트1: 인공지능은 인간의 학습 능력을 컴퓨터로 구현한 기술이다.
  텍스트2: 머신러닝은 데이터로부터 패턴을 학습하는 AI의 한 분야이다.
  텍스트3: 오늘 서울의 날씨는 맑고 기온은 25도이다.
  텍스트4: 파이썬은 데이터 과학에 널리 사용되는 프로그래밍 언어이다.

[유사도 해석]
  AI-머신러닝 유사도: 0.4824
  AI-날씨 유사도:     0.0036


# [7교시]

## 4. 벡터 검색 구현

질문을 벡터로 변환하고, 문서 벡터들과의 유사도를 계산하여 가장 관련성 높은 문서를 검색합니다.

In [43]:
# vector_search  벡터로된 문서장들을 서로 코사이유사도를 이용해서 높은순으로 정렬
def vector_search(query, documents, doc_embeddings, top_k=2):
    """질의에 대한 벡터 유사도 검색을 수행합니다."""
    query_embeded = get_embedding(query)
    simularity = []
    for i, doc_emb in enumerate(doc_embeddings):
        sim = cosine_similarity(query_embeded, doc_emb)
        simularity.append((i,sim,documents[i]))
    simularity.sort(key=lambda x : x[1], reverse=True) # 내림차순
    return simularity[:top_k]

# 검색 테스트
queries = [
    "AI와 기계학습의 관계는?",
    "프로그래밍 언어 추천",
    "내일 비가 오나요?"
]

for query in queries:
    print(f"\n{'=' * 60}")
    print(f"검색 질의: '{query}'")
    print("=" * 60)
    results = vector_search(query, texts, embeddings)
    for rank, (idx, sim, doc) in enumerate(results, 1):
        print(f"  {rank}위: [유사도: {sim:.4f}] {doc}")


검색 질의: 'AI와 기계학습의 관계는?'
  1위: [유사도: 0.5643] 인공지능은 인간의 학습 능력을 컴퓨터로 구현한 기술이다.
  2위: [유사도: 0.4389] 머신러닝은 데이터로부터 패턴을 학습하는 AI의 한 분야이다.

검색 질의: '프로그래밍 언어 추천'
  1위: [유사도: 0.4391] 파이썬은 데이터 과학에 널리 사용되는 프로그래밍 언어이다.
  2위: [유사도: 0.2198] 인공지능은 인간의 학습 능력을 컴퓨터로 구현한 기술이다.

검색 질의: '내일 비가 오나요?'
  1위: [유사도: 0.2695] 오늘 서울의 날씨는 맑고 기온은 25도이다.
  2위: [유사도: 0.0761] 머신러닝은 데이터로부터 패턴을 학습하는 AI의 한 분야이다.


## 5. ChromaDB 벡터 스토어 구축

ChromaDB는 오픈소스 벡터 데이터베이스로, 설치와 사용이 간편하며 인메모리/로컬 모드를 지원합니다. RAG 시스템의 벡터 스토어로 활용합니다.

In [44]:
# AI/RAG 관련 한국어 문서 10개
documents = [
    "RAG는 검색 증강 생성의 약자로, LLM의 한계를 보완하는 기술이다.",
    "벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 데이터베이스이다.",
    "임베딩은 텍스트를 수치 벡터로 변환하는 과정으로, 의미적 유사성을 보존한다.",
    "LangChain은 LLM 기반 애플리케이션을 구축하기 위한 프레임워크이다.",
    "트랜스포머는 어텐션 메커니즘을 기반으로 한 딥러닝 아키텍처이다.",
    "프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기법이다.",
    "파인튜닝은 사전 학습된 모델을 특정 도메인에 맞게 추가 학습하는 과정이다.",
    "청킹은 긴 문서를 작은 단위로 분할하는 과정으로, RAG의 핵심 전처리 단계이다.",
    "코사인 유사도는 두 벡터 간의 각도를 측정하여 유사성을 평가하는 방법이다.",
    "지식 그래프는 엔티티와 관계를 그래프 구조로 표현하는 지식 표현 방식이다."
]

import chromadb
# 인메모리방식
chroma_client = chromadb.Client()
# 컬렉션 생성
collection = chroma_client.get_or_create_collection(name='rag_demo',metadata={'hnsw:space':'cosine'})
# 문서 추가
collection.add(
    ids = [f'doc_{i}' for i in range(len(documents))],
    documents=documents,
    metadatas=[ { 'soucrce':'rag_demo','chunk_id':i }   for i in range(len(documents)) ]
)

# 유사도 검색
query = 'Rag 시스템에서 문서를 어떻게 검색하나요?'
results = collection.query(query_texts=[query], n_results=2)
for doc,dis in zip(results['documents'][0],results['distances'][0]):
    print(f'{1-dis} | {doc}')


0.7829707860946655 | 청킹은 긴 문서를 작은 단위로 분할하는 과정으로, RAG의 핵심 전처리 단계이다.
0.696295976638794 | RAG는 검색 증강 생성의 약자로, LLM의 한계를 보완하는 기술이다.


## 1. 이전 과정 회고: End-to-End 연결

지금까지 학습한 내용을 RAG 파이프라인의 각 단계에 매핑하면 다음과 같습니다.

```
                       RAG 파이프라인
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[1단계: Indexing]           [2단계: Retrieval]         [3단계: Generation]
 문서 → 청킹                    질문 → 임베딩               프롬프트 구성
 → 임베딩                       → 벡터 검색                 → LLM 호출(★이번)
 → 벡터 DB 저장                 → 관련 문서 반환             → 답변 생성
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

이번 에는 **Generation 단계** 를 추가하여 완전한 RAG 파이프라인을 구축합니다.

## 2. 수동 RAG 파이프라인 구축 (API 직접 호출)

LangChain을 사용하기 전에, OpenAI API를 직접 호출하여 **순수한 RAG 파이프라인** 을 먼저 구현합니다.

In [45]:
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# 1. 지식 베이스 문서 (인덱싱 단계)
knowledge_docs = [
    "RAG(Retrieval-Augmented Generation)는 2020년 Meta AI가 발표한 기술로, LLM과 정보 검색을 결합한다. 외부 문서에서 관련 정보를 검색하여 LLM의 컨텍스트로 제공함으로써 답변의 정확성을 높인다.",
    "벡터 데이터베이스는 텍스트 임베딩을 고차원 벡터로 저장하고, 코사인 유사도 기반의 빠른 검색을 지원한다. ChromaDB, FAISS, Pinecone이 대표적이다.",
    "청킹(Chunking)은 긴 문서를 적절한 크기의 조각으로 분할하는 과정이다. 고정 크기 청킹, 문장 기반 청킹, 재귀적 청킹 등의 전략이 있으며, 청크 크기는 검색 품질에 직접적인 영향을 미친다.",
    "임베딩(Embedding)은 텍스트를 의미적 정보를 보존하는 수치 벡터로 변환하는 과정이다. OpenAI의 text-embedding-3-small 모델은 1536차원의 벡터를 생성한다.",
    "LangChain은 LLM 기반 애플리케이션을 구축하기 위한 프레임워크로, 문서 로더, 텍스트 분할기, 임베딩, 벡터 스토어, 체인 등 다양한 컴포넌트를 제공한다.",
    "프롬프트 엔지니어링에서 RAG 시스템의 프롬프트는 시스템 지시사항, 검색된 컨텍스트, 사용자 질문으로 구성된다. 컨텍스트를 명확히 구분하고, 컨텍스트에 없는 내용은 답변하지 않도록 지시하는 것이 중요하다."
]

# 2. 임베딩 생성 (인덱싱 단계)
def get_embedding(text):
    response = client.embeddings.create(input=text, model="text-embedding-3-small")
    return response.data[0].embedding

doc_embeddings = [get_embedding(doc) for doc in knowledge_docs]
print(f"[인덱싱 완료] {len(knowledge_docs)}개 문서의 임베딩 생성 ({len(doc_embeddings[0])}차원)")

# 3. 검색 함수 (Retrieval 단계)
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def retrieve(query, top_k=3):
    query_emb = get_embedding(query)
    scores = [(i, cosine_similarity(query_emb, emb)) for i, emb in enumerate(doc_embeddings)]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [(knowledge_docs[i], sim) for i, sim in scores[:top_k]]

# 4. 생성 함수 (Generation 단계)
def generate_with_rag(query, retrieved_docs):
    context = "\n\n".join([f"[참고 문서 {i+1}] {doc}" for i, (doc, _) in enumerate(retrieved_docs)])
    
    messages = [
        {"role": "system", "content": "당신은 제공된 참고 문서를 바탕으로 정확하게 답변하는 AI 어시스턴트입니다. 참고 문서에 없는 내용은 '제공된 문서에서 해당 정보를 찾을 수 없습니다'라고 답변하세요."},
        {"role": "user", "content": f"참고 문서:\n{context}\n\n질문: {query}"}
    ]
    
    response = client.chat.completions.create(
        model="gpt-5.4-nano", messages=messages, temperature=0
    )
    return response.choices[0].message.content

# 5. 전체 파이프라인 실행
query = "RAG에서 청킹이 왜 중요한가요?"
print(f"\n질문: {query}")
print("=" * 60)

# Retrieval
retrieved = retrieve(query, top_k=3)
print("\n[검색된 문서]")
for i, (doc, sim) in enumerate(retrieved, 1):
    print(f"  {i}위 (유사도: {sim:.4f}): {doc[:60]}...")

# Generation
answer = generate_with_rag(query, retrieved)
print(f"\n[RAG 답변]")
print(answer)

[인덱싱 완료] 6개 문서의 임베딩 생성 (1536차원)

질문: RAG에서 청킹이 왜 중요한가요?

[검색된 문서]
  1위 (유사도: 0.4470): 프롬프트 엔지니어링에서 RAG 시스템의 프롬프트는 시스템 지시사항, 검색된 컨텍스트, 사용자 질문으로 구성된...
  2위 (유사도: 0.4461): 청킹(Chunking)은 긴 문서를 적절한 크기의 조각으로 분할하는 과정이다. 고정 크기 청킹, 문장 기반 ...
  3위 (유사도: 0.3309): RAG(Retrieval-Augmented Generation)는 2020년 Meta AI가 발표한 기술로,...

[RAG 답변]
RAG에서 **청킹(Chunking)** 이 중요한 이유는, 긴 문서를 **검색에 적합한 크기의 조각으로 나누어** 관련 정보를 검색할 수 있게 하기 때문입니다. 참고 문서에 따르면 **청크 크기는 검색 품질에 직접적인 영향을** 주며, 따라서 RAG에서 검색된 컨텍스트의 품질이 답변 정확성에 영향을 줄 수 있습니다.


## 3. RAG vs 순수 LLM 비교

RAG를 사용한 경우와 순수 LLM(컨텍스트 없음)을 사용한 경우의 답변 품질을 비교합니다.

In [46]:
# RAG vs 순수 LLM 비교 실험

def generate_without_rag(query):
    """순수 LLM (컨텍스트 없음)"""
    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{"role": "user", "content": query}],
        temperature=0
    )
    return response.choices[0].message.content

# 테스트 질문
test_query = "ChromaDB에서 코사인 유사도 검색을 설정하는 방법을 설명해주세요."

# 순수 LLM
print("=" * 60)
print(f"질문: {test_query}")
print("=" * 60)

print("\n[1] 순수 LLM 응답 (컨텍스트 없음):")
print("-" * 40)
llm_answer = generate_without_rag(test_query)
print(llm_answer[:300] + "..." if len(llm_answer) > 300 else llm_answer)

# RAG
print(f"\n\n[2] RAG 응답 (검색된 문서 기반):")
print("-" * 40)
retrieved = retrieve(test_query, top_k=3)
rag_answer = generate_with_rag(test_query, retrieved)
print(rag_answer[:300] + "..." if len(rag_answer) > 300 else rag_answer)

print(f"\n\n[비교 분석]")
print(f"  순수 LLM 응답 길이: {len(llm_answer)}자")
print(f"  RAG 응답 길이: {len(rag_answer)}자")
print(f"  검색된 참고 문서 수: {len(retrieved)}개")

질문: ChromaDB에서 코사인 유사도 검색을 설정하는 방법을 설명해주세요.

[1] 순수 LLM 응답 (컨텍스트 없음):
----------------------------------------
아래는 **ChromaDB에서 코사인 유사도(Cosine Similarity)로 검색**하도록 설정하는 대표적인 방법들입니다. (버전에 따라 API가 조금씩 다를 수 있지만, 핵심은 “거리/유사도 함수가 cosine이 되도록 컬렉션을 만들고, 쿼리 임베딩을 넣는 것”입니다.)

---

## 1) 컬렉션 생성 시 `metric`을 `cosine`으로 지정

ChromaDB에서 유사도(거리) 기준은 **컬렉션 생성 시** 정합니다.

```python
import chromadb

client = chromadb.Client()

c...


[2] RAG 응답 (검색된 문서 기반):
----------------------------------------
제공된 참고 문서 1에는 **ChromaDB가 코사인 유사도 기반의 빠른 검색을 지원한다**는 내용만 있고, **ChromaDB에서 코사인 유사도 검색을 “설정하는 구체적인 방법(예: 어떤 파라미터/옵션을 어떻게 지정하는지)”**에 대한 절차나 코드가 없습니다.  

따라서, 요청하신 “ChromaDB에서 코사인 유사도 검색을 설정하는 방법”은 **제공된 문서에서 해당 정보를 찾을 수 없습니다**.


[비교 분석]
  순수 LLM 응답 길이: 1842자
  RAG 응답 길이: 224자
  검색된 참고 문서 수: 3개


## 4. LangChain 기반 RAG 체인 구축

이번에는 LangChain의 컴포넌트를 활용하여 더 구조화된 RAG 파이프라인을 구축합니다.

In [48]:
!pip install langchain-community

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 33.9 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 48.7 MB/s  0:00:00

  Attempting uninstall: langchain-text-splitters

    Found existing installation: langchain-text-splitters 0.3.11

    Uninstalling langchain-text-splitters-0.3.11:

      Successfully uninstalled langchain-text-splitters-0.3.11

   ------------- -------------------------- 1/3 [langchain-classic]
   ------------- -------------------------- 1/3 [langchain-classic]
   ------------- -------------------------- 1/3 [langchain-classic]
   ------------- -------------------------- 1/3 [langchain-classic]
   ------------- -------------------------- 1/3 [langchain-classic]
   ------------- -------------------------- 1/3 [langchain-classic]
   ------------- -------------------------- 1/3 [langchain-classic]
   ---

In [49]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

# 1. 한국어 기술 문서 (더 긴 원본 문서)
raw_documents = [
    Document(page_content="RAG(Retrieval-Augmented Generation)는 2020년 Meta AI 연구팀이 발표한 기술이다. "
        "이 기술은 대규모 언어 모델의 생성 능력과 정보 검색 시스템을 결합한 하이브리드 접근법이다. "
        "기존의 LLM은 학습 데이터에만 의존하여 답변을 생성했기 때문에, 학습 이후의 정보에 대해서는 "
        "정확한 답변을 제공하기 어려웠다. RAG는 질문이 주어지면 먼저 외부 지식 소스에서 관련 문서를 "
        "검색하고, 검색된 문서를 컨텍스트로 활용하여 답변을 생성한다.",
        metadata={"source": "rag_overview", "topic": "RAG"}),
    Document(page_content="벡터 데이터베이스는 RAG 시스템의 핵심 인프라이다. 텍스트 데이터를 고차원 벡터로 변환하여 "
        "저장하고, 코사인 유사도를 기반으로 의미적으로 유사한 문서를 빠르게 검색할 수 있다. "
        "대표적인 벡터 DB로는 ChromaDB, FAISS, Pinecone, Weaviate가 있다. ChromaDB는 설치가 "
        "간편하고 인메모리 모드를 지원하여 프로토타이핑에 적합하다.",
        metadata={"source": "vector_db", "topic": "Vector DB"}),
    Document(page_content="프롬프트 엔지니어링은 LLM의 성능을 최적화하기 위한 핵심 기법이다. RAG 시스템에서의 "
        "프롬프트는 시스템 지시사항, 검색된 컨텍스트, 사용자 질문의 세 부분으로 구성된다. "
        "효과적인 RAG 프롬프트는 컨텍스트를 명확히 구분하고, 컨텍스트에 없는 내용은 답변하지 "
        "않도록 명시적으로 지시해야 한다. 이를 통해 환각을 줄이고 답변의 신뢰성을 높인다.",
        metadata={"source": "prompt_eng", "topic": "Prompt"})
]

# 2. 텍스트 분할 (청킹)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150, chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " "]
)
split_docs = text_splitter.split_documents(raw_documents)
print(f"[청킹] 원본 {len(raw_documents)}개 → 분할 {len(split_docs)}개")

# 3. 벡터 스토어 구축
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=split_docs, embedding=embeddings,
    collection_name="langchain_rag_demo"
)
print(f"[벡터 스토어] ChromaDB에 {len(split_docs)}개 청크 저장 완료")

# 4. 검색기 설정
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 5. RAG 프롬프트 템플릿
rag_prompt = ChatPromptTemplate.from_template(
    """당신은 제공된 참고 문서를 바탕으로 정확하게 답변하는 AI 어시스턴트입니다.
참고 문서에 없는 내용은 '제공된 문서에서 해당 정보를 찾을 수 없습니다'라고 답변하세요.

참고 문서:
{context}

질문: {question}

답변:"""
)

# 6. LLM 설정
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 7. RAG 체인 실행
query = "RAG 시스템에서 프롬프트를 어떻게 구성해야 하나요?"
print(f"\n질문: {query}")
print("=" * 60)

# 검색
retrieved_docs = retriever.invoke(query)
print(f"\n[검색 결과] {len(retrieved_docs)}개 청크")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"  {i}위: [{doc.metadata.get('topic', 'N/A')}] {doc.page_content[:60]}...")

# 컨텍스트 구성 및 생성
context = "\n\n".join([doc.page_content for doc in retrieved_docs])
formatted_prompt = rag_prompt.format(context=context, question=query)
response = llm.invoke(formatted_prompt)

print(f"\n[RAG 답변]")
print(response.content)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_31784\2279856076.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


[청킹] 원본 3개 → 분할 6개
[벡터 스토어] ChromaDB에 6개 청크 저장 완료

질문: RAG 시스템에서 프롬프트를 어떻게 구성해야 하나요?

[검색 결과] 3개 청크
  1위: [Prompt] 프롬프트 엔지니어링은 LLM의 성능을 최적화하기 위한 핵심 기법이다. RAG 시스템에서의 프롬프트는 시스템 ...
  2위: [Prompt] . 효과적인 RAG 프롬프트는 컨텍스트를 명확히 구분하고, 컨텍스트에 없는 내용은 답변하지 않도록 명시적으로...
  3위: [Vector DB] 벡터 데이터베이스는 RAG 시스템의 핵심 인프라이다. 텍스트 데이터를 고차원 벡터로 변환하여 저장하고, 코사...

[RAG 답변]
RAG 시스템에서 프롬프트는 시스템 지시사항, 검색된 컨텍스트, 사용자 질문의 세 부분으로 구성되어야 합니다. 효과적인 RAG 프롬프트는 컨텍스트를 명확히 구분하고, 컨텍스트에 없는 내용은 답변하지 않도록 명시적으로 지시해야 합니다. 이를 통해 환각을 줄이고 답변의 신뢰성을 높일 수 있습니다.


## 5. 다양한 질문으로 RAG 테스트

지식 베이스에 있는/없는 다양한 질문으로 RAG 시스템의 응답을 테스트합니다.

In [50]:
# 다양한 질문으로 RAG 테스트

test_questions = [
    "ChromaDB의 특징은 무엇인가요?",           # 지식 베이스에 있는 내용
    "RAG는 언제 누가 개발했나요?",              # 지식 베이스에 있는 내용
    "PyTorch와 TensorFlow의 차이는?",           # 지식 베이스에 없는 내용
]

for q in test_questions:
    print(f"\n{'=' * 60}")
    print(f"질문: {q}")
    print("-" * 60)
    
    docs = retriever.invoke(q)
    context = "\n\n".join([d.page_content for d in docs])
    prompt = rag_prompt.format(context=context, question=q)
    resp = llm.invoke(prompt)
    
    print(f"검색된 문서: {len(docs)}개")
    print(f"답변: {resp.content[:200]}")


질문: ChromaDB의 특징은 무엇인가요?
------------------------------------------------------------
검색된 문서: 3개
답변: ChromaDB는 설치가 간편하고 인메모리 모드를 지원하여 프로토타이핑에 적합한 벡터 데이터베이스입니다.

질문: RAG는 언제 누가 개발했나요?
------------------------------------------------------------
검색된 문서: 3개
답변: RAG(Retrieval-Augmented Generation)는 2020년 Meta AI 연구팀이 발표한 기술입니다.

질문: PyTorch와 TensorFlow의 차이는?
------------------------------------------------------------
검색된 문서: 3개
답변: 제공된 문서에서 해당 정보를 찾을 수 없습니다.


## 6. RAG 프롬프트 최적화

프롬프트 설계에 따라 RAG 응답의 품질이 크게 달라집니다. 다양한 프롬프트 전략을 비교합니다.

In [ ]:
# 프롬프트 변형 실험

prompt_variants = {
    "기본형": """참고 문서:\n{context}\n\n질문: {question}\n답변:""",
    "역할 부여형": """당신은 AI 기술 전문가입니다. 아래 참고 문서만을 근거로 답변하세요.
참고 문서에 없는 내용은 추측하지 마세요.

참고 문서:
{context}

질문: {question}
답변:""",
    "구조화된 형": """다음 지침에 따라 답변하세요:
1. 반드시 아래 참고 문서의 내용만 사용하세요.
2. 답변은 핵심 내용을 먼저 말하고 세부 사항을 추가하세요.
3. 참고 문서에 없는 내용은 '해당 정보가 문서에 없습니다'라고 하세요.

참고 문서:
{context}

질문: {question}
답변:"""
}

test_q = "RAG 기술의 핵심 원리를 설명해주세요."
docs = retriever.invoke(test_q)
context = "\n\n".join([d.page_content for d in docs])

for name, template in prompt_variants.items():
    print(f"\n{'=' * 60}")
    print(f"[프롬프트: {name}]")
    print("-" * 60)
    
    prompt = ChatPromptTemplate.from_template(template)
    formatted = prompt.format(context=context, question=test_q)
    resp = llm.invoke(formatted)
    
    print(resp.content[:250] + "..." if len(resp.content) > 250 else resp.content)


[프롬프트: 기본형]
------------------------------------------------------------
RAG(Retrieval-Augmented Generation) 기술의 핵심 원리는 대규모 언어 모델의 생성 능력과 정보 검색 시스템을 결합하여 보다 정확하고 신뢰성 있는 답변을 생성하는 것입니다. 이 기술은 다음과 같은 세 가지 주요 요소로 구성됩니다:

1. **정보 검색**: 사용자의 질문에 대한 관련 정보를 외부 데이터베이스나 문서에서 검색합니다. 이 과정에서 검색된 정보는 질문의 맥락을 제공하며, 모델이 보다 정확한 답변을 생성하는 데...

[프롬프트: 역할 부여형]
------------------------------------------------------------
RAG(Retrieval-Augmented Generation) 기술의 핵심 원리는 대규모 언어 모델의 생성 능력과 정보 검색 시스템을 결합하는 것입니다. 이 하이브리드 접근법은 사용자의 질문에 대해 관련된 정보를 검색하고, 그 정보를 바탕으로 보다 정확하고 신뢰성 있는 답변을 생성하는 데 중점을 둡니다. RAG 시스템은 프롬프트를 통해 시스템 지시사항, 검색된 컨텍스트, 사용자 질문의 세 가지 요소를 통합하여 효과적인 응답을 생성합니다. 이를...

[프롬프트: 구조화된 형]
------------------------------------------------------------
RAG 기술의 핵심 원리는 대규모 언어 모델의 생성 능력과 정보 검색 시스템을 결합하는 것입니다. 이를 통해 사용자가 질문할 때, 관련된 정보를 검색하고 그 정보를 바탕으로 보다 정확하고 신뢰성 있는 답변을 생성할 수 있습니다.

세부 사항으로는, RAG 시스템은 프롬프트 엔지니어링을 통해 성능을 최적화하며, 프롬프트는 시스템 지시사항, 검색된 컨텍스트, 사용자 질문의 세 부분으로 구성됩니다. 이러한 구조는 컨텍스트를 명확히 구분하고, 컨텍스트...


: 

## 7. 정리 및 핵심 요약

### 핵심 개념 정리

 개별 컴포넌트를 연결하여 **완전한 RAG 파이프라인** 을 구축했습니다.

| 구현 방식 | 특징 | 사용 시나리오 |
|----------|------|-------------|
| 수동 RAG | 각 단계를 직접 구현 | 원리 이해, 커스텀 파이프라인 |
| LangChain RAG | 프레임워크 컴포넌트 활용 | 빠른 프로토타이핑, 모듈식 구성 |

1. **End-to-End RAG**: 인덱싱(청킹→임베딩→저장) → 검색(질의→벡터검색) → 생성(프롬프트→LLM)의 3단계로 구성된다.

2. **RAG vs 순수 LLM**: RAG는 검색된 문서를 근거로 답변하여 환각을 줄이고, 지식 베이스에 없는 질문에 정직하게 답변할 수 있다.

3. **프롬프트 설계**: 역할 부여, 명시적 제한, 구조화된 지침이 RAG 답변의 품질을 크게 좌우한다.

---

# [8교시]

lm studio에서 다양한 모델 실습